<a href="https://colab.research.google.com/github/chuy-zip/LAB2_NLP/blob/main/MotorDBusquedaSem%C3%A1nticaLN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 Procesamiento de Lenguaje Natural

In [18]:
# !pip install sentence-transformers numpy scikit-learn pandas requests

In [19]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
import os

REPO_URL = "https://github.com/chuy-zip/LAB2_NLP.git"
REPO_DIR = "LAB2_NLP"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL}")
    print("Repositorio clonado correctamente.")
else:
    print("El repositorio ya existe, no se clona de nuevo.")

El repositorio ya existe, no se clona de nuevo.


#
  

In [21]:
import pandas as pd

CSV_PATH = "LAB2_NLP/grammys_1000_oraciones_es.csv"

df = pd.read_csv(CSV_PATH)
print(f"Columnas del CSV: {df.columns.tolist()}")
print(df.head(3))

# Tomar la columna de oraciones
COLUMNA = df.columns[4]
oraciones = df[COLUMNA].dropna().tolist()

print(f"\n Corpus cargado: {len(oraciones)} oraciones")
print(f"Ejemplo: {oraciones[0]}")

Columnas del CSV: ['id', 'year', 'song', 'artist', 'sentence']
   id  year                             song            artist  \
0   1  1959  Nel Blu Dipinto Di Blu (Volare)  Domenico Modugno   
1   2  1959  Nel Blu Dipinto Di Blu (Volare)  Domenico Modugno   
2   3  1959  Nel Blu Dipinto Di Blu (Volare)  Domenico Modugno   

                                            sentence  
0  En 1959, "Nel Blu Dipinto Di Blu (Volare)" obt...  
1  La obra "Nel Blu Dipinto Di Blu (Volare)", aso...  
2  Durante la ceremonia de los Grammy de 1959, "N...  

 Corpus cargado: 1000 oraciones
Ejemplo: En 1959, "Nel Blu Dipinto Di Blu (Volare)" obtuvo el Grammy a Canción del Año y fue interpretada por Domenico Modugno.


## Embeddings

In [22]:
# Se carga el modelo multilingüe de forma local (se descarga una sola vez)
modelo = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Generación de embeddings para todo el corpus
embeddings = modelo.encode(oraciones, show_progress_bar=True)

print(f"Shape de los embeddings: {embeddings.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Shape de los embeddings: (1000, 384)


### Función de búsqueda semántica (top-k coseno)

In [23]:
def buscar_semantica(consulta, top_k=3):
    # Busca las oraciones más similares a la consulta usando
    # similitud coseno sobre embeddings locales.

    embedding_consulta = modelo.encode([consulta])
    similitudes = cosine_similarity(embedding_consulta, embeddings)[0]
    indices_top = np.argsort(similitudes)[::-1][:top_k]

    resultados = []
    for idx in indices_top:
        resultados.append({
            "oracion": oraciones[idx],
            "similitud": round(float(similitudes[idx]), 4)
        })
    return resultados


def mostrar_resultados_semanticos(consulta, top_k=3):

    print(f"\nConsulta semántica: '{consulta}'")
    resultados = buscar_semantica(consulta, top_k)
    for i, r in enumerate(resultados, 1):
        print(f"  {i}. [{r['similitud']:.4f}] {r['oracion']}")

### Función de búsqueda por palabras clave (TF simple)

In [24]:
def tokenizar(texto):
    #Convierte texto a lista de tokens en minúsculas
    texto = texto.lower()
    return set(re.findall(r"\b\w+\b", texto, flags=re.UNICODE))


def buscar_keywords(consulta, top_k=3):

    # Busca oraciones que compartan más palabras clave con la consulta.
    # Score = cantidad de tokens en común (sin stopwords avanzadas).
    tokens_consulta = tokenizar(consulta)
    scores = []

    for oracion in oraciones:
        tokens_oracion = tokenizar(oracion)
        coincidencias = tokens_consulta & tokens_oracion
        scores.append(len(coincidencias))

    indices_top = np.argsort(scores)[::-1][:top_k]

    resultados = []
    for idx in indices_top:
        resultados.append({
            "oracion": oraciones[idx],
            "coincidencias": scores[idx]
        })
    return resultados



def mostrar_resultados_keywords(consulta, top_k=3):

    print(f"\nBúsqueda por keywords: '{consulta}'")

    resultados = buscar_keywords(consulta, top_k)
    for i, r in enumerate(resultados, 1):
        print(f"  {i}. [coincidencias: {r['coincidencias']}] {r['oracion']}")

In [25]:
consultas = [
    "¿Qué canción de Billie Eilish ganó el Grammy?",
    "canciones ganadoras de Adele en los premios",
    "¿Quién ganó canción del año en 2019?",
    "artistas que han ganado múltiples veces el Grammy",
    "reconocimientos de la Recording Academy a Bruno Mars",
]

for consulta in consultas:
    mostrar_resultados_semanticos(consulta, top_k=3)
    mostrar_resultados_keywords(consulta, top_k=3)
    print()


Consulta semántica: '¿Qué canción de Billie Eilish ganó el Grammy?'
  1. [0.7846] Billie Eilish estuvo vinculado al triunfo de "Bad Guy" en la categoría Canción del Año de los Grammy de 2020.
  2. [0.7800] Billie Eilish estuvo vinculado al triunfo de "What was I Made for" en la categoría Canción del Año de los Grammy de 2024.
  3. [0.7772] La canción "What was I Made for", interpretada por Billie Eilish, recibió uno de los principales Grammy en 2024.

Búsqueda por keywords: '¿Qué canción de Billie Eilish ganó el Grammy?'
  1. [coincidencias: 6] El premio Grammy a Canción del Año de 2020 correspondió a "Bad Guy", interpretada por Billie Eilish.
  2. [coincidencias: 6] La obra "What was I Made for", asociada con Billie Eilish, ganó la categoría Canción del Año en los Grammy de 2024.
  3. [coincidencias: 6] El premio Grammy a Canción del Año de 2024 correspondió a "What was I Made for", interpretada por Billie Eilish.


Consulta semántica: 'canciones ganadoras de Adele en los premios'
  